# 06 Sequence Models: RNN, LSTM, GRU & Seq2Seq

**Real-World Scenario**: Enterprise System Telemetry Stream Anomaly Classification System.

This notebook demonstrates building a PyTorch Bi-Directional LSTM Text Classifier, computing scalar LSTM gate updates step-by-step, saving model artifacts into a hidden `.model_cache/` directory, and executing a **Complete GenAI Anomaly Diagnosis LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Step-by-Step Scalar LSTM Gate Hand-Calculation

We compute single-step LSTM gate values ($f_t, i_t, \tilde{C}_t, C_t, o_t, h_t$) on a scalar feature $x_t=1.0$.

In [ ]:
import math

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

x_t = 1.0
h_prev = 0.5
c_prev = 2.0

# Weights
W_f, W_i, W_c, W_o = 0.5, 0.8, 0.6, 0.4

# 1. Compute Gate Activations
f_t = sigmoid(W_f * (h_prev + x_t))
i_t = sigmoid(W_i * (h_prev + x_t))
c_tilde = math.tanh(W_c * (h_prev + x_t))
o_t = sigmoid(W_o * (h_prev + x_t))

# 2. Compute Updated Cell State & Hidden State
c_t = (f_t * c_prev) + (i_t * c_tilde)
h_t = o_t * math.tanh(c_t)

print("=== Scalar LSTM Cell Hand-Calculation Results ===")
print(f"Forget Gate f_t:       {f_t:.4f}")
print(f"Input Gate i_t:        {i_t:.4f}")
print(f"Candidate Cell C~_t:   {c_tilde:.4f}")
print(f"Updated Cell State C_t:{c_t:.4f}")
print(f"Output Gate o_t:       {o_t:.4f}")
print(f"Updated Hidden State h_t:{h_t:.4f}")

## Step 3: PyTorch Bi-Directional LSTM Model Implementation & Artifact Persistence

In [ ]:
import torch
import torch.nn as nn

class PyTorchBiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        last_hidden = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)
        return self.fc(last_hidden)

# Model Setup
vocab_size, embed_dim, hidden_dim, num_classes = 1000, 64, 32, 2
model = PyTorchBiLSTMClassifier(vocab_size, embed_dim, hidden_dim, num_classes)

# Save PyTorch Weights to hidden cache folder
model_path = os.path.join(".model_cache", "bilstm_telemetry.pt")
torch.save(model.state_dict(), model_path)
print(f"=== PyTorch Bi-LSTM Model Weights Saved to Hidden Folder: {model_path} ===")

## Step 4: Model Forward Pass & Anomaly Log Sequence Inference

In [ ]:
# Dummy sequence input: Batch=2, SeqLen=10
sequence_tokens = torch.tensor([
    [10, 45, 12, 88, 90, 23, 11, 4, 2, 0],
    [99, 12, 40, 15, 66, 78, 12, 1, 0, 0]
])

model.eval()
with torch.no_grad():
    logits = model(sequence_tokens)
    probabilities = torch.softmax(logits, dim=1)

print("=== PyTorch Bi-LSTM Sequence Inference Output ===")
print("Logits Shape:       ", logits.shape)
print("Probabilities:      ", probabilities.numpy().round(4))
print("Sequence #1 Anomaly Status:", "ANOMALY DETECTED" if probabilities[0][1] > 0.5 else "NORMAL TELEMETRY")

## Step 5: Executing Complete GenAI Telemetry Anomaly Diagnosis LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

sample_telemetry_text = "Sequence Token IDs [10, 45, 12, 88, 90, 23]: High memory pressure on EKS cluster node followed by gRPC timeout on DB replica port 5432."

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise Infrastructure Reliability AI.
A Bi-Directional LSTM sequence model flagged telemetry sequence anomaly: '{telemetry}'.
Provide a 2-step root cause analysis and immediate remediation action plan.

Remediation Action Plan:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(telemetry=sample_telemetry_text))
    print("=== Complete GenAI Telemetry Anomaly Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")